# Notebook 01 — PPMI Data Access and Variable Inventory

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using Publicly Available Multimodal Data  
**Primary dataset:** Parkinson’s Progression Markers Initiative (PPMI)  
**Notebook status:** Data access verification and variable inventory only. No machine-learning model is trained in this notebook.

---

## Objective

This notebook verifies that the downloaded PPMI clinical/tabular data contain the minimum variables required to build a reproducible machine-learning project for Parkinson’s disease progression.

The notebook will:

1. Mount Google Drive in Google Colab.
2. Locate and extract the uploaded PPMI ZIP/CSV files.
3. Create a file-level inventory.
4. Verify `PATNO`, `EVENT_ID`, cohort definitions, and key outcome/predictor variables.
5. Summarize MDS-UPDRS Part III availability across visits.
6. Produce candidate follow-up windows for defining motor progression.
7. Save all inventory outputs to CSV files.

**Important:** This notebook does not share, publish, or export raw PPMI participant-level data outside your private Google Drive.

## Scientific Background

Parkinson’s disease is a progressive neurodegenerative disorder. For a publishable machine-learning study, the preferred scientific question is not simply “PD vs control classification”, but prediction of clinically meaningful progression over follow-up.

For this project, the initial primary outcome candidate is:

> **Motor progression**, operationalized as longitudinal change in **MDS-UPDRS Part III total score (`NP3TOT`)** from baseline to a future follow-up visit.

Potential baseline predictors include demographic variables, cohort/subgroup information, MDS-UPDRS Parts I–II, cognition, olfaction, sleep/RBD-related measures, depression/anxiety-related measures, medication-related variables, and clinical history.

PPMI group/subgroup definitions must be handled carefully because PPMI cohorts are heterogeneous and may differ at baseline and in longitudinal progression. Therefore, this notebook explicitly checks cohort labels and visit structure before any model development.

## Dataset Verification Plan

This notebook expects the following files in Google Drive.

### Minimum required files for Notebook 01

Upload these files to:

```text
MyDrive/PPMI_PD_Progression/data/raw/
```

Required:

```text
Motor___MDS-UPDRS.zip
Non-motor_Assessments.zip
Medical_History.zip
Participant_Status_30Jun2026.csv
```

Optional but recommended documentation files:

```text
Data_Dictionary_-__Annotated__23May2025.csv
Code_List_-__Annotated__23May2025.csv
PPMI Groups and Subgroups Guidance Document_18May2026.pdf
ppmi-publication-policy.pdf
PPMI Biomarkers Dashboard_20240819.xlsx
```

### Expected clinical files after ZIP extraction

The notebook will automatically search for these files after extraction:

- MDS-UPDRS Part I
- MDS-UPDRS Part I Patient Questionnaire
- MDS-UPDRS Part II Patient Questionnaire
- MDS-UPDRS Part III
- MDS-UPDRS Part IV
- Montreal Cognitive Assessment (MoCA)
- UPSIT
- REM Sleep Behavior Disorder Screening Questionnaire
- Geriatric Depression Scale
- Epworth Sleepiness Scale
- SCOPA-AUT
- State-Trait Anxiety Inventory
- LEDD Concomitant Medication Log
- Initiation of Dopaminergic Therapy
- PD Diagnosis History
- Primary Research Diagnosis
- Medical Conditions Log
- Vital Signs
- Neurological Exam
- Features of Parkinsonism
- Features of REM Behavior Disorder

## Code

Run the notebook from top to bottom. The only path you may need to edit is `PROJECT_DIR` if your Google Drive folder name is different.

In [ ]:
# ============================================================
# 01. Environment setup
# ============================================================

import os
import re
import sys
import json
import zipfile
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("Run time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# ============================================================
# 02. Mount Google Drive and define project folders
# ============================================================

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")
else:
    PROJECT_DIR = Path.cwd() / "PPMI_PD_Progression"

RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
EXTRACT_DIR = INTERIM_DIR / "extracted_csv"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "notebook_01_variable_inventory"
DOCS_DIR = PROJECT_DIR / "docs"

for folder in [PROJECT_DIR, RAW_DIR, INTERIM_DIR, EXTRACT_DIR, OUTPUT_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_DIR:", RAW_DIR)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("\nIMPORTANT: Upload the raw PPMI CSV/ZIP files to RAW_DIR exactly as shown above.")


In [ ]:
# ============================================================
# 03. Confirm expected raw files are available
# ============================================================

raw_files = sorted([p for p in RAW_DIR.glob("*") if p.is_file()])

print(f"Files found in RAW_DIR: {len(raw_files)}")
for p in raw_files:
    print(f"- {p.name} ({p.stat().st_size/1024/1024:.2f} MB)")

expected_minimum_keywords = ["Motor", "Non-motor", "Medical_History", "Participant_Status"]
missing_keywords = []
for kw in expected_minimum_keywords:
    if not any(kw.lower() in p.name.lower() for p in raw_files):
        missing_keywords.append(kw)

if missing_keywords:
    print("\nWARNING: These expected file keywords were not found in RAW_DIR:")
    print(missing_keywords)
    print("\nExpected upload location:")
    print(RAW_DIR)
    print("\nExpected minimum files:")
    for name in [
        "Motor___MDS-UPDRS.zip",
        "Non-motor_Assessments.zip",
        "Medical_History.zip",
        "Participant_Status_30Jun2026.csv",
    ]:
        print("-", name)
else:
    print("\nMinimum expected raw files appear to be present.")

# Optional diagnostic search: look for likely files elsewhere in MyDrive.
if IN_COLAB and len(raw_files) == 0:
    print("\nDiagnostic search: looking for ZIP/CSV files elsewhere in MyDrive...")
    mydrive = Path("/content/drive/MyDrive")
    likely = []
    patterns = ["*MDS*UPDRS*.zip", "*Non*motor*.zip", "*Medical*History*.zip", "*Participant*Status*.csv", "*.zip", "*.csv"]
    for pattern in patterns:
        likely.extend(mydrive.rglob(pattern))
    likely = sorted(set([p for p in likely if p.is_file()]))[:80]
    print(f"Candidate files found elsewhere in MyDrive: {len(likely)}")
    for p in likely[:40]:
        print("-", p)
    if len(likely) > 40:
        print("... more files found. Move/copy the required four files into RAW_DIR.")


In [ ]:
# ============================================================
# OPTIONAL: Upload files directly from your computer into RAW_DIR
# Use this only if RAW_DIR is empty or you cannot find the Drive folder.
# After upload finishes, rerun cells 03, 04, and 05.
# ============================================================

RUN_DIRECT_UPLOAD = False  # Change to True only when you want to upload files manually.

if RUN_DIRECT_UPLOAD:
    if not IN_COLAB:
        raise RuntimeError("Direct upload is available only in Google Colab.")
    from google.colab import files
    uploaded = files.upload()
    for fname, content in uploaded.items():
        out_path = RAW_DIR / fname
        with open(out_path, "wb") as f:
            f.write(content)
        print("Saved:", out_path, f"({out_path.stat().st_size/1024/1024:.2f} MB)")
    print("Upload complete. Now rerun cells 03, 04, and 05.")


In [ ]:
# ============================================================
# 04. Extract ZIP files safely
# ============================================================

def safe_extract_zip(zip_path: Path, destination_root: Path) -> Path:
    """Extract a zip file into a dedicated folder named after the zip stem."""
    target_dir = destination_root / zip_path.stem
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.namelist()
        for member in members:
            member_path = target_dir / member
            # zip-slip protection
            if not str(member_path.resolve()).startswith(str(target_dir.resolve())):
                raise RuntimeError(f"Unsafe path in ZIP: {member}")
        zf.extractall(target_dir)
    return target_dir

zip_files = sorted(RAW_DIR.glob("*.zip"))
print(f"ZIP files found in RAW_DIR: {len(zip_files)}")

for z in zip_files:
    out_dir = safe_extract_zip(z, EXTRACT_DIR)
    print(f"Extracted: {z.name} -> {out_dir}")

# List extracted top-level contents for verification.
if EXTRACT_DIR.exists():
    extracted_items = sorted(EXTRACT_DIR.rglob("*"))[:80]
    print(f"\nExtracted folder items shown: {len(extracted_items)}")
    for p in extracted_items[:40]:
        if p.is_file():
            print("-", p.relative_to(PROJECT_DIR))


In [ ]:
# ============================================================
# 05. Locate all CSV files in raw and extracted folders
# ============================================================

csv_files = sorted(set(list(RAW_DIR.rglob("*.csv")) + list(EXTRACT_DIR.rglob("*.csv"))))

print(f"Total CSV files found: {len(csv_files)}")
for p in csv_files:
    rel = p.relative_to(PROJECT_DIR) if PROJECT_DIR in p.parents else p
    print("-", rel)

if len(csv_files) == 0:
    print("\nERROR: No CSV files were found in RAW_DIR or EXTRACT_DIR.")
    print("This usually means the ZIP/CSV files were not uploaded to the expected Google Drive folder.")
    print("\nExpected folder:")
    print(RAW_DIR)
    print("\nFix option A: Move the files in Google Drive to the folder above, then rerun cells 02–05.")
    print("Fix option B: Run the optional upload cell below and upload the four required files directly.")
    raise FileNotFoundError("No CSV files found in RAW_DIR or EXTRACT_DIR.")


In [ ]:
# ============================================================
# 06. Utility functions
# ============================================================

def normalize_text(x: str) -> str:
    """Normalize a filename or label for flexible matching."""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def read_csv_robust(path: Path, nrows=None) -> pd.DataFrame:
    """Read CSV robustly with common encodings."""
    encodings = ["utf-8", "utf-8-sig", "latin1"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, low_memory=False, encoding=enc, nrows=nrows)
        except Exception as e:
            last_error = e
    raise last_error


def first_existing_column(df: pd.DataFrame, candidates):
    """Return first matching column from candidates, or None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None


def save_table(df: pd.DataFrame, filename: str) -> Path:
    """Save a dataframe to OUTPUT_DIR."""
    out = OUTPUT_DIR / filename
    df.to_csv(out, index=False)
    print(f"Saved: {out}")
    return out


def summarize_missing(series: pd.Series):
    total = len(series)
    missing = int(series.isna().sum())
    return missing, round(100 * missing / total, 2) if total else np.nan

In [ ]:
# ============================================================
# 07. File-level inventory
# ============================================================

inventory_rows = []

for path in csv_files:
    try:
        df = read_csv_robust(path)
        cols = list(df.columns)
        row = {
            "file_name": path.name,
            "relative_path": str(path.relative_to(PROJECT_DIR)) if PROJECT_DIR in path.parents else str(path),
            "n_rows": len(df),
            "n_columns": len(cols),
            "has_PATNO": "PATNO" in cols,
            "has_EVENT_ID": "EVENT_ID" in cols,
            "n_unique_PATNO": df["PATNO"].nunique(dropna=True) if "PATNO" in cols else np.nan,
            "n_unique_EVENT_ID": df["EVENT_ID"].nunique(dropna=True) if "EVENT_ID" in cols else np.nan,
            "duplicate_PATNO_EVENT_rows": int(df.duplicated(["PATNO", "EVENT_ID"]).sum()) if {"PATNO", "EVENT_ID"}.issubset(cols) else np.nan,
            "columns_preview": ", ".join(cols[:20])
        }
    except Exception as e:
        row = {
            "file_name": path.name,
            "relative_path": str(path),
            "n_rows": np.nan,
            "n_columns": np.nan,
            "has_PATNO": False,
            "has_EVENT_ID": False,
            "n_unique_PATNO": np.nan,
            "n_unique_EVENT_ID": np.nan,
            "duplicate_PATNO_EVENT_rows": np.nan,
            "columns_preview": f"ERROR: {e}"
        }
    inventory_rows.append(row)

file_inventory = pd.DataFrame(inventory_rows).sort_values("file_name")
display(file_inventory)
save_table(file_inventory, "01_file_inventory.csv")

In [ ]:
# ============================================================
# 08. Flexible file matching for required datasets
# ============================================================

expected_patterns = {
    "participant_status": ["participant_status"],
    "mds_updrs_part_i": ["mds_updrs_part_i_30", "mds_updrs_part_i", "part_i_30"],
    "mds_updrs_part_i_patient": ["part_i_patient", "patient_questionnaire", "np1p"],
    "mds_updrs_part_ii": ["part_ii", "np2", "motor_aspects"],
    "mds_updrs_part_iii": ["part_iii", "np3", "motor_examination"],
    "mds_updrs_part_iv": ["part_iv", "np4", "motor_complications"],
    "moca": ["moca", "montreal_cognitive"],
    "upsit": ["upsit", "smell_identification"],
    "rbd_questionnaire": ["rem_sleep_behavior_disorder_screening", "rbd"],
    "features_rbd": ["features_of_rem_behavior"],
    "gds": ["geriatric_depression"],
    "epworth": ["epworth"],
    "scopa_aut": ["scopa"],
    "stai": ["state_trait_anxiety", "stai"],
    "ledd": ["ledd_concomitant", "ledd"],
    "dopaminergic_therapy": ["initiation_of_dopaminergic"],
    "pd_diagnosis_history": ["pd_diagnosis_history"],
    "primary_research_diagnosis": ["primary_research_diagnosis"],
    "medical_conditions": ["medical_conditions_log"],
    "vital_signs": ["vital_signs"],
    "neurological_exam": ["neurological_exam"],
    "features_parkinsonism": ["features_of_parkinsonism"],
}

csv_norm = {p: normalize_text(p.name) for p in csv_files}

matches = {}
for key, patterns in expected_patterns.items():
    matched = []
    for p, norm in csv_norm.items():
        if any(pattern in norm for pattern in patterns):
            matched.append(p)
    matches[key] = sorted(matched, key=lambda x: len(str(x)))

match_table = []
for key, paths in matches.items():
    match_table.append({
        "expected_dataset": key,
        "n_matches": len(paths),
        "selected_file": paths[0].name if paths else None,
        "all_matches": " | ".join([p.name for p in paths]) if paths else None
    })

match_df = pd.DataFrame(match_table)
display(match_df)
save_table(match_df, "02_expected_dataset_file_matches.csv")

# Helper dictionary to load selected files later
selected_paths = {key: (paths[0] if paths else None) for key, paths in matches.items()}

In [ ]:
# ============================================================
# 09. Load selected datasets
# ============================================================

loaded = {}
for key, path in selected_paths.items():
    if path is None:
        continue
    try:
        loaded[key] = read_csv_robust(path)
        print(f"Loaded {key}: {path.name} -> {loaded[key].shape}")
    except Exception as e:
        print(f"ERROR loading {key}: {path} -> {e}")

print("\nDatasets loaded:", list(loaded.keys()))

In [ ]:
# ============================================================
# 10. Participant Status verification
# ============================================================

if "participant_status" not in loaded:
    raise FileNotFoundError("Participant Status file was not found. Upload Participant_Status_30Jun2026.csv and rerun.")

patients = loaded["participant_status"].copy()

required_patient_cols = ["PATNO", "COHORT", "COHORT_DEFINITION"]
missing_patient_cols = [c for c in required_patient_cols if c not in patients.columns]
if missing_patient_cols:
    raise ValueError(f"Participant Status is missing required columns: {missing_patient_cols}")

print("Participant Status shape:", patients.shape)
print("Unique PATNO:", patients["PATNO"].nunique())
print("Duplicate PATNO rows:", patients.duplicated("PATNO").sum())

cohort_distribution = (
    patients.groupby(["COHORT", "COHORT_DEFINITION"], dropna=False)
    .agg(n_participants=("PATNO", "nunique"))
    .reset_index()
    .sort_values("COHORT")
)

display(cohort_distribution)
save_table(cohort_distribution, "03_cohort_distribution_from_participant_status.csv")

if "ENROLL_AGE" in patients.columns:
    miss_n, miss_pct = summarize_missing(patients["ENROLL_AGE"])
    print(f"ENROLL_AGE missing: {miss_n} ({miss_pct}%)")

In [ ]:
# ============================================================
# 11. Variable availability matrix
# ============================================================

key_variables = {
    "participant_status": ["PATNO", "COHORT", "COHORT_DEFINITION", "ENROLL_DATE", "ENROLL_STATUS", "ENROLL_AGE",
                           "ENRLLRRK2", "ENRLGBA", "ENRLSNCA", "ENRLPRKN", "ENRLHPSM", "ENRLRBD"],
    "mds_updrs_part_i": ["PATNO", "EVENT_ID", "INFODT", "NP1RTOT"],
    "mds_updrs_part_i_patient": ["PATNO", "EVENT_ID", "INFODT", "NP1PTOT"],
    "mds_updrs_part_ii": ["PATNO", "EVENT_ID", "INFODT", "NP2PTOT"],
    "mds_updrs_part_iii": ["PATNO", "EVENT_ID", "INFODT", "PDTRTMNT", "PDSTATE", "PDMEDYN", "DBSYN", "NP3TOT", "NHY"],
    "mds_updrs_part_iv": ["PATNO", "EVENT_ID", "INFODT", "NP4TOT"],
    "moca": ["PATNO", "EVENT_ID", "INFODT", "MCATOT"],
    "upsit": ["PATNO", "EVENT_ID", "INFODT", "TOTAL_CORRECT"],
    "rbd_questionnaire": ["PATNO", "EVENT_ID", "INFODT", "DRMVIVID", "DRMFIGHT", "PARKISM"],
    "features_rbd": ["PATNO", "EVENT_ID", "INFODT", "RBDDIAG", "RBDPSG"],
    "gds": ["PATNO", "EVENT_ID", "INFODT"],
    "epworth": ["PATNO", "EVENT_ID", "INFODT"],
    "scopa_aut": ["PATNO", "EVENT_ID", "INFODT"],
    "stai": ["PATNO", "EVENT_ID", "INFODT"],
    "ledd": ["PATNO", "EVENT_ID", "LEDTRT", "LEDD", "STARTDT", "STOPDT"],
    "dopaminergic_therapy": ["PATNO", "EVENT_ID", "INFODT", "DOPTHERST"],
    "pd_diagnosis_history": ["PATNO", "EVENT_ID", "INFODT", "SXDT", "PDDXDT", "DOMSIDE"],
    "primary_research_diagnosis": ["PATNO", "EVENT_ID", "INFODT", "PRIMDIAG", "DXLVL"],
    "medical_conditions": ["PATNO", "EVENT_ID", "INFODT", "MHCAT", "MHTERM", "RESOLVD"],
    "vital_signs": ["PATNO", "EVENT_ID", "INFODT", "WGTKG", "HTCM", "SYSSUP", "DIASUP", "SYSSTND", "DIASTND"],
    "neurological_exam": ["PATNO", "EVENT_ID", "INFODT"],
    "features_parkinsonism": ["PATNO", "EVENT_ID", "INFODT", "FEATBRADY", "FEATRIGID", "FEATTREMOR"],
}

availability_rows = []
for dataset_key, variables in key_variables.items():
    df = loaded.get(dataset_key)
    if df is None:
        for var in variables:
            availability_rows.append({
                "dataset": dataset_key,
                "variable": var,
                "dataset_loaded": False,
                "present": False,
                "n_non_missing": np.nan,
                "percent_non_missing": np.nan,
                "n_unique_values": np.nan
            })
        continue
    for var in variables:
        present = var in df.columns
        if present:
            n_non_missing = int(df[var].notna().sum())
            pct_non_missing = round(100 * n_non_missing / len(df), 2) if len(df) else np.nan
            n_unique = int(df[var].nunique(dropna=True))
        else:
            n_non_missing = np.nan
            pct_non_missing = np.nan
            n_unique = np.nan
        availability_rows.append({
            "dataset": dataset_key,
            "variable": var,
            "dataset_loaded": True,
            "present": present,
            "n_non_missing": n_non_missing,
            "percent_non_missing": pct_non_missing,
            "n_unique_values": n_unique
        })

variable_availability = pd.DataFrame(availability_rows)
display(variable_availability)
save_table(variable_availability, "04_variable_availability_matrix.csv")

In [ ]:
# ============================================================
# 12. Visit/event availability by dataset
# ============================================================

visit_rows = []
for key, df in loaded.items():
    if "EVENT_ID" not in df.columns or "PATNO" not in df.columns:
        continue
    event_counts = (
        df.groupby("EVENT_ID", dropna=False)
        .agg(n_rows=("PATNO", "size"), n_unique_patients=("PATNO", "nunique"))
        .reset_index()
    )
    event_counts.insert(0, "dataset", key)
    visit_rows.append(event_counts)

visit_availability = pd.concat(visit_rows, ignore_index=True) if visit_rows else pd.DataFrame()

display(visit_availability.head(50))
save_table(visit_availability, "05_visit_availability_by_dataset.csv")

In [ ]:
# ============================================================
# 13. MDS-UPDRS Part III primary outcome inventory
# ============================================================

if "mds_updrs_part_iii" not in loaded:
    raise FileNotFoundError("MDS-UPDRS Part III file was not found. This file is required for the primary outcome inventory.")

mds3 = loaded["mds_updrs_part_iii"].copy()

required_mds3_cols = ["PATNO", "EVENT_ID", "NP3TOT"]
missing_mds3_cols = [c for c in required_mds3_cols if c not in mds3.columns]
if missing_mds3_cols:
    raise ValueError(f"MDS-UPDRS Part III is missing required columns: {missing_mds3_cols}")

# Merge cohort labels for descriptive inventory.
mds3_cohort = mds3.merge(
    patients[["PATNO", "COHORT", "COHORT_DEFINITION"]],
    on="PATNO",
    how="left"
)

print("MDS-UPDRS Part III shape:", mds3.shape)
print("Unique participants:", mds3["PATNO"].nunique())
print("Rows with non-missing NP3TOT:", int(mds3["NP3TOT"].notna().sum()))
print("Duplicate PATNO/EVENT_ID rows:", int(mds3.duplicated(["PATNO", "EVENT_ID"]).sum()))

np3_event = (
    mds3_cohort.loc[mds3_cohort["NP3TOT"].notna()]
    .groupby(["EVENT_ID", "COHORT", "COHORT_DEFINITION"], dropna=False)
    .agg(
        n_rows=("PATNO", "size"),
        n_unique_patients=("PATNO", "nunique"),
        mean_np3tot=("NP3TOT", "mean"),
        sd_np3tot=("NP3TOT", "std")
    )
    .reset_index()
)

# A simple order for common PPMI visit codes.
visit_order_map = {
    "SC": -1, "BL": 0,
    "V01": 1, "V02": 2, "V03": 3, "V04": 4, "V05": 5, "V06": 6, "V07": 7, "V08": 8,
    "V09": 9, "V10": 10, "V11": 11, "V12": 12, "V13": 13, "V14": 14, "V15": 15, "V16": 16,
    "V17": 17, "V18": 18, "V19": 19, "V20": 20
}
np3_event["visit_order"] = np3_event["EVENT_ID"].map(visit_order_map).fillna(999)
np3_event = np3_event.sort_values(["COHORT", "visit_order", "EVENT_ID"])

display(np3_event.head(80))
save_table(np3_event, "06_np3tot_event_availability_by_cohort.csv")

In [ ]:
# ============================================================
# 14. Candidate follow-up windows for motor progression
# ============================================================

# For this inventory, we count unique participants who have non-missing NP3TOT at BL and at each candidate follow-up.
# We do NOT finalize the outcome definition here because duplicate PATNO/EVENT_ID rows and treatment state require preprocessing decisions.

candidate_followups = ["V04", "V06", "V08", "V10", "V12", "V14", "V17"]

# Restrict to PD participants for the primary analysis candidate.
pd_patnos = set(patients.loc[patients["COHORT_DEFINITION"].eq("Parkinson's Disease"), "PATNO"])
mds3_pd = mds3.loc[mds3["PATNO"].isin(pd_patnos) & mds3["NP3TOT"].notna()].copy()

# Count unique PATNO per visit after non-missing NP3TOT.
patients_by_visit = {
    event: set(mds3_pd.loc[mds3_pd["EVENT_ID"].eq(event), "PATNO"].dropna().unique())
    for event in ["BL"] + candidate_followups
}

candidate_rows = []
for fu in candidate_followups:
    baseline_set = patients_by_visit.get("BL", set())
    follow_set = patients_by_visit.get(fu, set())
    paired = baseline_set.intersection(follow_set)
    candidate_rows.append({
        "cohort": "Parkinson's Disease",
        "baseline_event": "BL",
        "followup_event": fu,
        "n_with_baseline_np3tot": len(baseline_set),
        "n_with_followup_np3tot": len(follow_set),
        "n_paired_baseline_followup": len(paired),
        "comment": "Candidate only; final outcome requires duplicate/treatment-state preprocessing."
    })

outcome_candidate_pairs = pd.DataFrame(candidate_rows)
display(outcome_candidate_pairs)
save_table(outcome_candidate_pairs, "07_candidate_motor_progression_followup_windows.csv")

In [ ]:
# ============================================================
# 15. Baseline availability of core predictors among PD participants
# ============================================================

core_predictors = {
    "mds_updrs_part_i": "NP1RTOT",
    "mds_updrs_part_i_patient": "NP1PTOT",
    "mds_updrs_part_ii": "NP2PTOT",
    "mds_updrs_part_iii": "NP3TOT",
    "mds_updrs_part_iv": "NP4TOT",
    "moca": "MCATOT",
    "upsit": "TOTAL_CORRECT",
    "vital_signs": "WGTKG",
    "pd_diagnosis_history": "PDDXDT",
    "primary_research_diagnosis": "PRIMDIAG",
    "dopaminergic_therapy": "DOPTHERST",
}

baseline_rows = []
for dataset_key, variable in core_predictors.items():
    df = loaded.get(dataset_key)
    if df is None:
        baseline_rows.append({
            "dataset": dataset_key,
            "variable": variable,
            "dataset_loaded": False,
            "present": False,
            "n_pd_patients_with_BL_nonmissing": np.nan,
            "n_pd_patients_with_SC_nonmissing": np.nan,
            "n_pd_patients_with_BL_or_SC_nonmissing": np.nan
        })
        continue
    present = variable in df.columns and "PATNO" in df.columns and "EVENT_ID" in df.columns
    if present:
        df_pd = df[df["PATNO"].isin(pd_patnos)].copy()
        bl = set(df_pd.loc[df_pd["EVENT_ID"].eq("BL") & df_pd[variable].notna(), "PATNO"].unique())
        sc = set(df_pd.loc[df_pd["EVENT_ID"].eq("SC") & df_pd[variable].notna(), "PATNO"].unique())
        bl_or_sc = bl.union(sc)
        baseline_rows.append({
            "dataset": dataset_key,
            "variable": variable,
            "dataset_loaded": True,
            "present": True,
            "n_pd_patients_with_BL_nonmissing": len(bl),
            "n_pd_patients_with_SC_nonmissing": len(sc),
            "n_pd_patients_with_BL_or_SC_nonmissing": len(bl_or_sc)
        })
    else:
        baseline_rows.append({
            "dataset": dataset_key,
            "variable": variable,
            "dataset_loaded": True,
            "present": False,
            "n_pd_patients_with_BL_nonmissing": np.nan,
            "n_pd_patients_with_SC_nonmissing": np.nan,
            "n_pd_patients_with_BL_or_SC_nonmissing": np.nan
        })

baseline_predictor_availability = pd.DataFrame(baseline_rows)
display(baseline_predictor_availability)
save_table(baseline_predictor_availability, "08_baseline_core_predictor_availability_PD.csv")

In [ ]:
# ============================================================
# 16. Data quality flags
# ============================================================

quality_flags = []

def add_flag(item, status, detail):
    quality_flags.append({"qc_item": item, "status": status, "detail": detail})

add_flag(
    "Participant Status present",
    "PASS" if "participant_status" in loaded else "FAIL",
    "Master participant table is required."
)

add_flag(
    "Participant Status unique PATNO",
    "PASS" if patients.duplicated("PATNO").sum() == 0 else "WARNING",
    f"Duplicate PATNO rows: {int(patients.duplicated('PATNO').sum())}"
)

add_flag(
    "MDS-UPDRS Part III present",
    "PASS" if "mds_updrs_part_iii" in loaded else "FAIL",
    "Required for primary motor progression outcome."
)

add_flag(
    "NP3TOT present",
    "PASS" if "NP3TOT" in mds3.columns else "FAIL",
    "MDS-UPDRS Part III total score variable."
)

add_flag(
    "Duplicate PATNO/EVENT_ID in MDS-UPDRS Part III",
    "WARNING" if mds3.duplicated(["PATNO", "EVENT_ID"]).sum() > 0 else "PASS",
    f"Duplicate rows: {int(mds3.duplicated(['PATNO','EVENT_ID']).sum())}. This requires preprocessing before outcome construction."
)

for var in ["MCATOT", "TOTAL_CORRECT"]:
    present_anywhere = any(var in df.columns for df in loaded.values())
    add_flag(
        f"{var} availability",
        "PASS" if present_anywhere else "WARNING",
        f"{var} found in loaded datasets: {present_anywhere}"
    )

# Candidate follow-up recommendation based on paired sample size only.
if not outcome_candidate_pairs.empty:
    best_by_n = outcome_candidate_pairs.sort_values("n_paired_baseline_followup", ascending=False).iloc[0]
    add_flag(
        "Largest paired follow-up candidate by sample size",
        "INFO",
        f"{best_by_n['followup_event']} has {int(best_by_n['n_paired_baseline_followup'])} paired PD participants with BL and follow-up NP3TOT. Scientific choice still requires follow-up-duration justification."
    )

add_flag(
    "No ML modeling performed",
    "PASS",
    "This notebook only performs access verification and variable inventory."
)

qc_df = pd.DataFrame(quality_flags)
display(qc_df)
save_table(qc_df, "09_quality_control_checklist.csv")

## Scientific Interpretation

Interpret the generated tables as follows:

1. **`01_file_inventory.csv`** confirms whether every uploaded file was readable and whether it contains `PATNO` and `EVENT_ID`.
2. **`03_cohort_distribution_from_participant_status.csv`** defines the available participant groups.
3. **`04_variable_availability_matrix.csv`** confirms whether the expected variables are present and how complete they are.
4. **`06_np3tot_event_availability_by_cohort.csv`** shows whether MDS-UPDRS Part III is available longitudinally.
5. **`07_candidate_motor_progression_followup_windows.csv`** identifies candidate follow-up visits for defining motor progression.
6. **`09_quality_control_checklist.csv`** highlights issues that must be solved before preprocessing and modeling.

A high-quality Q1-level project should not proceed to feature engineering or model training until these outputs show that the target outcome can be defined reproducibly.

In [ ]:
# ============================================================
# 17. Create a human-readable summary report
# ============================================================

summary_lines = []
summary_lines.append("Notebook 01 — PPMI Data Access and Variable Inventory")
summary_lines.append(f"Run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Project directory: {PROJECT_DIR}")
summary_lines.append("")
summary_lines.append(f"CSV files found: {len(csv_files)}")
summary_lines.append(f"Datasets loaded: {len(loaded)}")
summary_lines.append("")
summary_lines.append("Participant Status")
summary_lines.append(f"- Rows: {patients.shape[0]}")
summary_lines.append(f"- Columns: {patients.shape[1]}")
summary_lines.append(f"- Unique PATNO: {patients['PATNO'].nunique()}")
summary_lines.append("")
summary_lines.append("MDS-UPDRS Part III")
summary_lines.append(f"- Rows: {mds3.shape[0]}")
summary_lines.append(f"- Unique participants: {mds3['PATNO'].nunique()}")
summary_lines.append(f"- Rows with non-missing NP3TOT: {int(mds3['NP3TOT'].notna().sum())}")
summary_lines.append(f"- Duplicate PATNO/EVENT_ID rows: {int(mds3.duplicated(['PATNO','EVENT_ID']).sum())}")
summary_lines.append("")
summary_lines.append("Candidate motor progression follow-up windows")
summary_lines.append(outcome_candidate_pairs.to_string(index=False))
summary_lines.append("")
summary_lines.append("QC checklist")
summary_lines.append(qc_df.to_string(index=False))

summary_text = "\n".join(summary_lines)
summary_path = OUTPUT_DIR / "10_notebook_01_summary_report.txt"
summary_path.write_text(summary_text, encoding="utf-8")

print(summary_text)
print(f"\nSaved summary report: {summary_path}")

## Quality Control Checklist

Before moving to Notebook 02, confirm the following:

- [ ] `Participant_Status` is present and `PATNO` is unique.
- [ ] MDS-UPDRS Part III is present and includes `NP3TOT`.
- [ ] Baseline `BL` and at least one follow-up visit have sufficient `NP3TOT` coverage.
- [ ] Duplicate `PATNO`/`EVENT_ID` rows in MDS-UPDRS Part III are understood before outcome construction.
- [ ] Candidate predictors are restricted to baseline or pre-outcome information only.
- [ ] Future visit data are not used as predictors.
- [ ] Treatment state and medication timing are handled explicitly before defining the final outcome.
- [ ] PPMI cohort/subgroup definitions are considered in the statistical analysis plan.
- [ ] No raw participant-level PPMI data are placed in public repositories.

## Expected Output

After running this notebook, the following files should be created in:

```text
MyDrive/PPMI_PD_Progression/outputs/notebook_01_variable_inventory/
```

```text
01_file_inventory.csv
02_expected_dataset_file_matches.csv
03_cohort_distribution_from_participant_status.csv
04_variable_availability_matrix.csv
05_visit_availability_by_dataset.csv
06_np3tot_event_availability_by_cohort.csv
07_candidate_motor_progression_followup_windows.csv
08_baseline_core_predictor_availability_PD.csv
09_quality_control_checklist.csv
10_notebook_01_summary_report.txt
```

These outputs are the formal decision documents for whether we can proceed to Notebook 02.

---

## Decision Gate Before Notebook 02

Do not proceed to preprocessing/modeling until the following decisions are made from Notebook 01 outputs:

1. Which cohort is primary? Usually **Parkinson’s Disease only**.
2. Which follow-up visit defines the primary outcome? Candidate: **V06 or V08**, depending on paired sample size and scientific follow-up window.
3. How will duplicate MDS-UPDRS Part III rows per participant/visit be handled?
4. Will medication/treatment state be included, excluded, or stratified?
5. Which baseline predictors are sufficiently complete?
6. Which variables must be excluded to avoid data leakage?